# Text Summarization — Fine-Tuning PEGASUS on SAMSum

Fine-tune `google/pegasus-cnn_dailymail` on the [SAMSum](https://huggingface.co/datasets/samsum) dialogue-summarization dataset and evaluate using ROUGE scores.

> **GPU recommended.** PEGASUS is a ~570M-parameter model; training on CPU is very slow.

## Installation

In [ ]:
!pip install -q "transformers[sentencepiece]" datasets evaluate rouge_score accelerate

: 

## Imports & Device Setup

In [ ]:
import torch
import pandas as pd
import evaluate
from tqdm import tqdm

from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    pipeline,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device.upper()}")

---
## 1. Load Model & Tokenizer

In [ ]:
MODEL_CKPT = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CKPT).to(device)

---
## 2. Load & Explore Dataset

In [ ]:
dataset_samsum = load_dataset("samsum")
print(dataset_samsum)

In [ ]:
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]
print(f"Split lengths:  {split_lengths}")
print(f"Features:       {dataset_samsum['train'].column_names}")

In [ ]:
sample = dataset_samsum["test"][1]
print("--- Dialogue ---")
print(sample["dialogue"])
print("\n--- Summary ---")
print(sample["summary"])

---
## 3. Tokenize Dataset

In [ ]:
def convert_examples_to_features(batch):
    input_encodings = tokenizer(
        batch["dialogue"], max_length=1024, truncation=True
    )
    # text_target is the modern replacement for the deprecated as_target_tokenizer context manager
    target_encodings = tokenizer(
        text_target=batch["summary"], max_length=128, truncation=True
    )
    return {
        "input_ids":      input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels":         target_encodings["input_ids"],
    }

dataset_samsum_pt = dataset_samsum.map(
    convert_examples_to_features,
    batched=True,
    remove_columns=["id", "dialogue", "summary"],
)
print(dataset_samsum_pt)

---
## 4. Fine-Tune with Trainer

In [ ]:
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [ ]:
training_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model_pegasus,
    args=training_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["train"],
    eval_dataset=dataset_samsum_pt["validation"],
)

trainer.train()

---
## 5. Evaluate with ROUGE

In [ ]:
def generate_batch_sized_chunks(elements, batch_size):
    for i in range(0, len(elements), batch_size):
        yield elements[i : i + batch_size]


def calculate_metric_on_test_ds(
    dataset,
    metric,
    model,
    tokenizer,
    batch_size=16,
    column_text="dialogue",
    column_summary="summary",
):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches  = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)
    ):
        inputs = tokenizer(
            article_batch,
            max_length=1024,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=8,
            max_new_tokens=128,
            early_stopping=True,
        )

        decoded = [
            tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            for s in summaries
        ]

        metric.add_batch(predictions=decoded, references=target_batch)

    return metric.compute()

In [ ]:
rouge_metric = evaluate.load("rouge")
rouge_names  = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

score = calculate_metric_on_test_ds(
    dataset_samsum["test"][:10],
    rouge_metric,
    trainer.model,
    tokenizer,
    batch_size=2,
)

rouge_dict = {rn: round(score[rn], 4) for rn in rouge_names}
pd.DataFrame(rouge_dict, index=["pegasus-samsum"])

---
## 6. Save & Load

In [ ]:
trainer.model.save_pretrained("pegasus-samsum-model")
tokenizer.save_pretrained("pegasus-samsum-tokenizer")
print("Model and tokenizer saved.")

In [ ]:
tokenizer_loaded = AutoTokenizer.from_pretrained("pegasus-samsum-tokenizer")
model_loaded     = AutoModelForSeq2SeqLM.from_pretrained("pegasus-samsum-model").to(device)

---
## 7. Inference

In [ ]:
summarizer = pipeline(
    "summarization",
    model=model_loaded,
    tokenizer=tokenizer_loaded,
    device=0 if device == "cuda" else -1,
)

sample_text = dataset_samsum["test"][0]["dialogue"]
reference   = dataset_samsum["test"][0]["summary"]

gen_kwargs  = {"length_penalty": 0.8, "num_beams": 8, "max_new_tokens": 128}
prediction  = summarizer(sample_text, **gen_kwargs)[0]["summary_text"]

print("--- Dialogue ---")
print(sample_text)
print("\n--- Reference Summary ---")
print(reference)
print("\n--- Model Summary ---")
print(prediction)

---
## 8. ROUGE Comparison — Fine-Tuned vs Base Model

In [ ]:
rouge_base = evaluate.load("rouge")
score_base = calculate_metric_on_test_ds(
    dataset_samsum["test"][:10],
    rouge_base,
    model_pegasus,
    tokenizer,
    batch_size=2,
)

rouge_ft = evaluate.load("rouge")
score_ft = calculate_metric_on_test_ds(
    dataset_samsum["test"][:10],
    rouge_ft,
    trainer.model,
    tokenizer,
    batch_size=2,
)

pd.DataFrame(
    {
        "Base": {rn: round(score_base[rn], 4) for rn in rouge_names},
        "Fine-Tuned": {rn: round(score_ft[rn], 4) for rn in rouge_names},
    }
).T